# Matrice di frequenza

ogni vettore rappresenta le parole nella frase 

vettori verticali uguali avranno significato simile

In [1]:
from sklearn.feature_extraction.text import CountVectorizer

# Dataset sintetico (integrato nel codice per semplicità)
corpus = [
    'Il gatto insegue il topo.',
    'Il topo mangia il formaggio.',
    'Il gatto dorme sul divano.'
]

# Inizializziamo il vettorizzatore
vectorizer = CountVectorizer()

# Creiamo la matrice di frequenza
X = vectorizer.fit_transform(corpus)

# Visualizziamo i risultati
print("Vocabolario:", vectorizer.get_feature_names_out())
print("Matrice di Frequenza:\n", X.toarray())

Vocabolario: ['divano' 'dorme' 'formaggio' 'gatto' 'il' 'insegue' 'mangia' 'sul' 'topo']
Matrice di Frequenza:
 [[0 0 0 1 2 1 0 0 1]
 [0 0 1 0 2 0 1 0 1]
 [1 1 0 1 1 0 0 1 0]]


# Tf-idf

### 1. TF (Term Frequency) — "L'importanza locale"

Misura quanto una parola è frequente nel singolo documento che stiamo analizzando.

* **Logica:** Se una parola appare molte volte in un testo, probabilmente è un argomento centrale di quel testo.
* **Esempio:** In un articolo di cucina, la parola "forno" apparirà spesso (TF alto).

### 2. IDF (Inverse Document Frequency) — "La rarità globale"

Misura quanto la parola è rara o comune in tutti i documenti della tua collezione (corpus).

* **Logica:** Se una parola appare in quasi tutti i documenti (come "il", "che", "è"), non è utile per distinguere un testo dall'altro. L'IDF "punisce" queste parole comuni e "premia" quelle uniche.
* **Esempio:** La parola "forno" è meno comune della parola "il". Quindi "forno" ha un IDF più alto.

---

### La Formula e il Bilanciamento

Il punteggio finale è il prodotto dei due valori:

$$TF-IDF = TF \times IDF$$

**In sintesi, il TF-IDF risponde a questa domanda:**

> *"Questa parola è frequente qui (TF alto), ma è abbastanza rara negli altri documenti (IDF alto) da essere considerata una parola chiave distintiva?"*

* **Punteggio Alto:** La parola è una "parola chiave" (es. "fotovoltaico" in un articolo sull'energia).
* **Punteggio Basso:** La parola è rumore grammaticale (es. "un", "per") oppure è troppo comune per essere specifica (es. "cosa").


In [4]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Definiamo un corpus più ampio con temi distinti
# Inseriamo intenzionalmente molte parole comuni come "il", "è", "di", "che"
corpus = [
    """L'intelligenza artificiale rappresenta il pilastro fondamentale della tecnologia moderna, 
    influenzando lo sviluppo di nuovi algoritmi e software avanzati che simulano le capacità umane. 
    Nel prossimo futuro, l'apprendimento automatico e le reti neurali trasformeranno radicalmente 
    ogni settore industriale, migliorando l'efficienza dei sistemi digitali.""",

    """La cucina italiana è celebre in tutto il mondo non solo per il gusto, ma soprattutto perché 
    il cibo è il risultato di una tradizione millenaria basata sulla qualità delle materie prime 
    e sulla biodiversità. Ogni regione d'Italia offre ricette uniche che spaziano dalla pasta fresca 
    alla pizza, mantenendo standard di eccellenza gastronomica.""",

    """Il software di ultima generazione integrato in questa nuova tecnologia permette di automatizzare 
    processi complessi e di cucinare piatti sofisticati con l'ausilio dell'intelligenza artificiale. 
    L'innovazione tecnologica sta portando strumenti intelligenti direttamente nelle case delle persone, 
    semplificando la preparazione del cibo quotidiano.""",

    """Roma, conosciuta come la Città Eterna, è la capitale d'Italia e rappresenta il cuore pulsante 
    della tradizione politica, storica e culturale del paese. Camminando tra i suoi monumenti antichi 
    si respira la storia millenaria dell'impero, mentre oggi la città funge da centro istituzionale 
    fondamentale per la democrazia moderna."""
]

# 2. Inizializziamo il Vectorizer 
# NOTA: Non usiamo stop_words per far vedere alla classe che il TF-IDF 
# "punisce" naturalmente le parole comuni senza bisogno di cancellarle a mano.
vectorizer = TfidfVectorizer()

# 3. Generiamo la matrice TF-IDF
tfidf_matrix = vectorizer.fit_transform(corpus)

# 4. Estraiamo i nomi delle parole (il nostro vocabolario)
feature_names = vectorizer.get_feature_names_out()

# 5. Creiamo un DataFrame per visualizzare i "vettori" di ogni documento
df_tfidf = pd.DataFrame(
    tfidf_matrix.toarray(), 
    columns=feature_names, 
    index=["Doc 1 (Tech)", "Doc 2 (Cibo)", "Doc 3 (Mix)", "Doc 4 (Storia)"]
)

# Mostriamo le prime 15 parole per brevità o quelle più significative
print("--- ANALISI DEI VETTORI TF-IDF ---")
# Trasponiamo (.T) per vedere meglio la lista di parole e i relativi punteggi
print(df_tfidf.T.round(3))

# 6. Analisi specifica per la classe: confronto pesi
print("\n--- CONFRONTO DIRETTO DEI PUNTEGGI ---")
parole_test = ["il", "intelligenza", "cucina", "roma", "di"]
for parola in parole_test:
    if parola in feature_names:
        # Prendiamo il valore massimo che quella parola raggiunge nel corpus
        max_score = df_tfidf[parola].max()
        print(f"Parola: '{parola.ljust(12)}' | Punteggio Max: {max_score:.3f}")

--- ANALISI DEI VETTORI TF-IDF ---
               Doc 1 (Tech)  Doc 2 (Cibo)  Doc 3 (Mix)  Doc 4 (Storia)
algoritmi             0.165         0.000        0.000           0.000
alla                  0.000         0.144        0.000           0.000
antichi               0.000         0.000        0.000           0.145
apprendimento         0.165         0.000        0.000           0.000
artificiale           0.130         0.000        0.130           0.000
...                     ...           ...          ...             ...
tutto                 0.000         0.144        0.000           0.000
ultima                0.000         0.000        0.165           0.000
umane                 0.165         0.000        0.000           0.000
una                   0.000         0.144        0.000           0.000
uniche                0.000         0.144        0.000           0.000

[137 rows x 4 columns]

--- CONFRONTO DIRETTO DEI PUNTEGGI ---
Parola: 'il          ' | Punteggio Max: 0.301
Par

## Limite di TF-IDF

In italiano ci sono parole che hanno significato diverso a seconda del contesto, come "pesca"

## Word Embeddings: Il Significato diventa Geometria

Mentre il TF-IDF isola le parole in colonne separate (trattandole come entità indipendenti), gli **Embedding** le proiettano in uno spazio multidimensionale dove la **distanza** tra due punti ne rappresenta la somiglianza concettuale.

---

### 1. Vettori Densi vs Vettori Sparsi

* **TF-IDF (Vettore Sparso):** Una tabella enorme, piena di zeri, dove ogni parola è un'isola. Se due parole sono sinonimi, il TF-IDF non lo sa: per lui sono solo due colonne diverse.
* **Embedding (Vettore Denso):** Ogni parola è una sequenza fissa di numeri (es. 128, 300 o 768 dimensioni). Questi numeri non sono conteggi, ma "coordinate" che definiscono la posizione della parola in una mappa del significato.

### 2. Vicinanza Semantica

In questo spazio geometrico, parole che appaiono in contesti simili finiscono vicine.

* *Esempio:* I vettori di **"pizza"** e **"pasta"** saranno geometricamente molto vicini tra loro, mentre saranno lontanissimi dal vettore di **"algoritmo"**.

### 3. Algebra Relazionale

La caratteristica più rivoluzionaria degli embedding è la capacità di risolvere analogie tramite il calcolo vettoriale. Poiché le relazioni tra parole sono codificate come distanze e direzioni, possiamo scrivere:

$$\vec{v}_{\text{Re}} - \vec{v}_{\text{Uomo}} + \vec{v}_{\text{Donna}} \approx \vec{v}_{\text{Regina}}$$

Sottraendo il vettore "Uomo" dal vettore "Re", isoliamo il concetto di "regalità"; aggiungendo il vettore "Donna", ci spostiamo matematicamente verso il punto dello spazio che identifica la "Regina".

### 4. Come vengono creati?

I modelli (come **Word2Vec**, **GloVe** o i più moderni **Transformer**) imparano queste coordinate analizzando enormi quantità di testo. La logica è: *"Conoscerai una parola dalla compagnia che frequenta"*. Se "gatto" e "cane" appaiono spesso vicino a verbi come "mangiare", "dormire" o "giocare", i loro vettori verranno trascinati vicini durante l'addestramento.

In [5]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 1. Preparazione del testo
corpus = [
    """L'intelligenza artificiale rappresenta il pilastro fondamentale della tecnologia moderna, 
    influenzando lo sviluppo di nuovi algoritmi e software avanzati che simulano le capacità umane. 
    Nel prossimo futuro, l'apprendimento automatico e le reti neurali trasformeranno radicalmente 
    ogni settore industriale, migliorando l'efficienza dei sistemi digitali.""",

    """La cucina italiana è celebre in tutto il mondo non solo per il gusto, ma soprattutto perché 
    il cibo è il risultato di una tradizione millenaria basata sulla qualità delle materie prime 
    e sulla biodiversità. Ogni regione d'Italia offre ricette uniche che spaziano dalla pasta fresca 
    alla pizza, mantenendo standard di eccellenza gastronomica.""",

    """Il software di ultima generazione integrato in questa nuova tecnologia permette di automatizzare 
    processi complessi e di cucinare piatti sofisticati con l'ausilio dell'intelligenza artificiale. 
    L'innovazione tecnologica sta portando strumenti intelligenti direttamente nelle case delle persone, 
    semplificando la preparazione del cibo quotidiano.""",

    """Roma, conosciuta come la Città Eterna, è la capitale d'Italia e rappresenta il cuore pulsante 
    della tradizione politica, storica e culturale del paese. Camminando tra i suoi monumenti antichi 
    si respira la storia millenaria dell'impero, mentre oggi la città funge da centro istituzionale 
    fondamentale per la democrazia moderna."""
]

# 2. Tokenizzazione (assegniamo un numero a ogni parola)
tokenizer = Tokenizer()
tokenizer.fit_on_texts(corpus)
word_index = tokenizer.word_index
vocab_size = len(word_index) + 1  # +1 per il padding (lo zero)

# Trasformiamo le frasi in sequenze di numeri
sequences = tokenizer.texts_to_sequences(corpus)
# Aggiungo alla fine delle frase corte degli 0 per renderle tutte della stessa lunghezza 
# le gpu sono ottimizzate per parallelizzazione di calcoli su matrici identiche, quindi devo rendere le matrici identiche per aumentare la velocità di calcolo
data = pad_sequences(sequences, padding='post')

# 3. Creazione del Modello con il Layer di Embedding
# input_dim = dimensione del vocabolario
# output_dim = dimensione del vettore (quante coordinate ha ogni parola)
# input_length = lunghezza della frase
model = Sequential()
model.add(Embedding(input_dim=vocab_size, output_dim=4, input_length=data.shape[1]))

# 4. Vediamo cosa succede!
# Passiamo i dati al modello per ottenere gli embedding
embeddings = model.predict(data)

print(f"Forma dei dati in ingresso (Indici): {data.shape}") # (3 frasi, 6 parole)
print(f"Forma dell'output (Embedding): {embeddings.shape}") # (3 frasi, 6 parole, 4 coordinate)

# Stampiamo il risultato per la prima parola della prima frase
print("\nEsempio di trasformazione:")
print(f"Frase originale: {corpus[0]}")
print(f"Indici numerici: {data[0]}")
print(f"Vettore (Embedding) della prima parola:\n{embeddings[0][0]}")

c:\Users\mbagn\anaconda3\envs\pytorch\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 186ms/step
Forma dei dati in ingresso (Indici): (4, 53)
Forma dell'output (Embedding): (4, 53, 4)

Esempio di trasformazione:
Frase originale: L'intelligenza artificiale rappresenta il pilastro fondamentale della tecnologia moderna, 
    influenzando lo sviluppo di nuovi algoritmi e software avanzati che simulano le capacità umane. 
    Nel prossimo futuro, l'apprendimento automatico e le reti neurali trasformeranno radicalmente 
    ogni settore industriale, migliorando l'efficienza dei sistemi digitali.
Indici numerici: [26  6  7  1 27  8  9 10 11 28 29 30  3 31 32  4 12 33 13 34 14 35 36 37
 38 39 40 41  4 14 42 43 44 45 15 46 47 48 49 50 51 52  0  0  0  0  0  0
  0  0  0  0  0]
Vettore (Embedding) della prima parola:
[-0.0152285  -0.03351687 -0.03286703 -0.03850685]


# Dato il modello di embedding come misuriamo la similarità tra due parole?

In [10]:
import torch
from sentence_transformers import SentenceTransformer, util

# Carichiamo il modello di alta qualità (solo inglese)
model = SentenceTransformer('all-mpnet-base-v2', device='cpu')

# 1. Analizziamo la similitudine (Cosine Similarity)
words = ["Cat", "Dog", "Gas Station"]
word_vectors = model.encode(words)

sim_cat_dog = util.cos_sim(word_vectors[0], word_vectors[1])
sim_cat_gas = util.cos_sim(word_vectors[0], word_vectors[2])

print(f"--- TEST SIMILITUDINE ---")
print(f"Similitudine Cat-Dog: {sim_cat_dog.item():.4f} (Alta: sono entrambi animali domestici)")
print(f"Similitudine Cat-Gas Station: {sim_cat_gas.item():.4f} (Bassa: contesti diversi)\n")

# 2. TEST ALGEBRA VETTORIALE (King - Man + Woman = Queen)
print(f"--- TEST ALGEBRA DEI CONCETTI ---")
concepts = ["King", "Man", "Woman", "Queen"]
v = {word: model.encode(word) for word in concepts}

# Eseguiamo l'operazione: Risultato = Re - Uomo + Donna
target_vector = v["King"] - v["Man"] + v["Woman"]

# Calcoliamo quanto il punto matematico ottenuto sia vicino a "Queen"
analogy_score = util.cos_sim(target_vector, v["Queen"])

print(f"Risultato dell'operazione 'King - Man + Woman':")
print(f"Vicinanza alla parola 'Queen': {analogy_score.item():.4f}")

c:\Users\mbagn\anaconda3\envs\pytorch\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\mbagn\.cache\huggingface\hub\models--sentence-transformers--all-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 582.98it/s, Materializing param=poo

--- TEST SIMILITUDINE ---
Similitudine Cat-Dog: 0.6081 (Alta: sono entrambi animali domestici)
Similitudine Cat-Gas Station: 0.2199 (Bassa: contesti diversi)

--- TEST ALGEBRA DEI CONCETTI ---
Risultato dell'operazione 'King - Man + Woman':
Vicinanza alla parola 'Queen': 0.5093


# Come si usano in azienda?

Trasformo i documenti in vettori

Trasformo una domanda in vettore

E in base alla similarità trovo il documento che risponde alla domanda

Anche se le keyword di ricerca non sono perfette

In [11]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

# 1. Il nostro "Database" di documenti
documents = [
    "L'intelligenza artificiale automatizza i compiti ripetitivi usando algoritmi.",
    "La lasagna è un piatto tipico della tradizione culinaria italiana.",
    "Il Colosseo è uno dei monumenti più famosi della città di Roma.",
    "I modelli linguistici come i Transformer capiscono il contesto del testo.",
    "La pizza margherita è fatta con pomodoro, mozzarella e basilico."
]

# 2. Carichiamo il modello per trasformare i testi in vettori
model = SentenceTransformer('all-mpnet-base-v2', device='cpu')
document_embeddings = model.encode(documents)

# 3. Creiamo l'Indice FAISS
# Usiamo un indice di tipo 'FlatL2' (misura la distanza Euclidea tra vettori)
dimension = document_embeddings.shape[1] # 768 per questo modello
index = faiss.IndexFlatL2(dimension)

# Aggiungiamo i vettori dei documenti all'indice
index.add(np.array(document_embeddings).astype('float32'))

# 4. Facciamo una ricerca (Query)
query = "Qual è il cibo più famoso in Italia?"
query_embedding = model.encode([query])

# Cerchiamo i 2 documenti più simili (k=2)
distances, indices = index.search(np.array(query_embedding).astype('float32'), k=2)

print(f"Domanda: {query}\n")
print("Risultati trovati:")
for i, idx in enumerate(indices[0]):
    print(f"{i+1}. {documents[idx]} (Distanza: {distances[0][i]:.4f})")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 766.85it/s, Materializing param=pooler.dense.weight]                        
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Domanda: Qual è il cibo più famoso in Italia?

Risultati trovati:
1. Il Colosseo è uno dei monumenti più famosi della città di Roma. (Distanza: 0.7030)
2. La pizza margherita è fatta con pomodoro, mozzarella e basilico. (Distanza: 0.8808)
